
# Классификация ориентации текстового кропа (0° / 180°)

**Задача:** для каждого кропа с текстом, найденного детектором текста OCR-пайплайна, предсказать
`p_180` — вероятность того, что текст на кропе повёрнут на 180° 

**Метрика:** `1 - Brier Score`. То есть задача - не обычная бинарная классификация, важна калибровка вероятности: модель должна быть уверена там, где действительно уверена, и  выдавать значения около 0.5 в неоднозначных ситуациях

## Особенность решения
Размеченных обучающих данных нет, ручная разметка запрещена, поэтому решение построено на синтетически сгенерированных данных: сами рендерим текст в правильной ("upright") ориентации, метка 0/180 создаётся автоматически случайным переворотом на этапе загрузки батча. Из-за отсутствия размеченного реального теста для валидации используется
дополнительная self-consistency проверка (она не требует знания истинных меток).

## Структура ноутбука
0. Установка зависимостей, импорты, фиксация random seed
1. Конфигурация всех путей и гиперпараметров в одном месте (CFG)
2. Распаковка тестового датасета (test.zip)
3. Генератор синтетических данных (шрифты, фон, стилизация текста, аугментации)
4. Dataset / DataLoader с авто-разметкой ориентации
5. Модель - MobileNetV3-Small (количество параметров 1,518,881)
6. Обучение (AMP, CUDA, label smoothing, early stopping)
7. Калибровка вероятностей, которая напрямую улучшает Brier score
8. Инференс на тестовом датасете, получаем submission.csv + self-consistency проверка качества
9. Текстовое описание подхода

## 0. Установка зависимостей, импорты

In [ ]:

# Установка PyTorch для CUDA 12.1 и вспомогательные библиотеки.
# Если PyTorch/CUDA уже настроены в окружении, эту ячейку можно пропустить.
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q
!pip install timm albumentations opencv-python-headless pillow scikit-learn pandas tqdm -q


In [ ]:
import os
import io
import glob
import math
import random
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageOps

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import models, transforms

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score

from tqdm.auto import tqdm

# Фиксация random seed
# задание явно требует зафиксировать seed, если в решении есть случайность (а у нас везде: генерация синтетики, разбиение train/val, порядок батчей, случайный переворот кропов на 180° и т.д.)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Обучение происходило на CUDA
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))

## 1. Конфигурация

In [ ]:
# Все пути и гиперпараметры собраны в одном месте, чтобы не искать их по всему ноутбуку при воспроизведении решения
class CFG:
    PROJECT_DIR   = Path("./orientation_project")
    SYNTH_DIR     = PROJECT_DIR / "synthetic"           # сюда сохраняются синтетические upright-кропы
    FONTS_DIR     = Path("./fonts")                     # .ttf/.otf шрифты для рендеринга синтетики (кириллица + латиница)
    TEST_DIR      = Path("./test_boxes")                # сюда распаковывается тестовый датасет (20 000 боксов)
    TEST_CSV      = Path("./sample_submission.csv")     # пример сабмишна — эталонный список image_id и формат
    OUT_SUBMISSION = Path("./submission.csv")           # итоговый файл с предсказаниями

    N_SYNTH_SAMPLES = 60000   # количество генерируемых синтетических кропов
    IMG_H = 64                # высота входа модели (увеличена с исходных 48, что улучшило качество на реальных данных)
    IMG_W = 256                # ширина входа модели (была 192, увеличена вместе с IMG_H)

    BATCH_SIZE = 256
    EPOCHS = 20                # верхняя граница; реально используется early stopping
    LR = 3e-4
    WEIGHT_DECAY = 5e-4        # L2-регуляризация. Увеличена относительно дефолта, чтобы снизить переобучение под синтетику
    NUM_WORKERS = 4            # число воркеров DataLoader
    VAL_FRACTION = 0.1         # доля данных под hold-out валидацию 

    BACKBONE = "mobilenet_v3_small"   # Попытка использования large-модели выдала такой результат: итоговый балл (1 − Brier) = 0.74093604; ошибка Брайера = 0.25906396 против итоговый балл (1 − Brier) = 0.70082343; ошибка Брайера = 0.29917657 при использовании модели small 

# Сздание нужных директорий заранее
CFG.PROJECT_DIR.mkdir(exist_ok=True, parents=True)
CFG.SYNTH_DIR.mkdir(exist_ok=True, parents=True)
CFG.FONTS_DIR.mkdir(exist_ok=True, parents=True)

## 2. Распаковка тестового датасета

In [ ]:
# Распаковываем test.zip в папку CFG.TEST_DIR
import zipfile
import shutil

zip_path = Path("./test.zip")  
if zip_path.exists():
    with zipfile.ZipFile(zip_path, "r") as zf:
        names = zf.namelist()
        print("Файлов в архиве:", len(names))
        print(names[:10])
        zf.extractall(CFG.TEST_DIR)
    print("Распаковано в:", CFG.TEST_DIR.resolve())
else:
    #или можно воспользоваться командой shutil.unpack_archive("test.zip", CFG.TEST_DIR)
    print("test.zip не найден по указанному пути")


## 3. Синтетическая генерация данных

Рендерим строку текста в правильной ориентации с большим разнообразием визуальных факторов (шрифты, фон, стилизация, геометрические и фотографические искажения) так, чтобы синтетика как можно ближе приближалась к реальным фото объявлений/вывесок/табличек. Метка 0/180 создаётся позже — автоматическим случайным переворотом каждого кропа на этапе загрузки батча (OrientationDataset)

Набор аугментаций в разделе ниже - результат нескольких итераций: после каждого обучения смотрела на self-consistency ошибку модели на реальном test.zip и на худшие по ней кропы глазами, после чего добавляла эффект, воспроизводящий замеченный пробел (насыщенные цветные вывески - HSV-фон; гравировка - отдельная "металлик" ветка + эмбосс и т.д.)

In [ ]:

# Скачивание набора открытых шрифтов с Google Fonts 
import urllib.request
from pathlib import Path

CFG.FONTS_DIR.mkdir(exist_ok=True, parents=True)

FONT_URLS = {
    # декоративные/жирные  (стиль вывесок, баннеров, логотипов)
    "Anton.ttf":            "ofl/anton/Anton-Regular.ttf",
    "BebasNeue.ttf":        "ofl/bebasneue/BebasNeue-Regular.ttf",
    "Righteous.ttf":        "ofl/righteous/Righteous-Regular.ttf",
    "PermanentMarker.ttf":  "ofl/permanentmarker/PermanentMarker-Regular.ttf",
    "Pacifico.ttf":         "ofl/pacifico/Pacifico-Regular.ttf",
    "Bangers.ttf":          "ofl/bangers/Bangers-Regular.ttf",
    "Oswald-Bold.ttf":      "ofl/oswald/Oswald%5Bwght%5D.ttf",
    "Rubik-Bold.ttf":       "ofl/rubik/Rubik%5Bwght%5D.ttf",
    "OpenSans.ttf":         "ofl/opensans/OpenSans%5Bwdth%2Cwght%5D.ttf",
    "Comfortaa.ttf":        "ofl/comfortaa/Comfortaa%5Bwght%5D.ttf",
    "Merriweather.ttf":     "ofl/merriweather/Merriweather%5Bopsz%2Cwdth%2Cwght%5D.ttf",
    "Lobster.ttf":          "ofl/lobster/Lobster-Regular.ttf",
    "Fjalla.ttf":           "ofl/fjallaone/FjallaOne-Regular.ttf",
    "BlackOpsOne.ttf":      "ofl/blackopsone/BlackOpsOne-Regular.ttf",
    # базовые sans-serif с поддержкой кириллицы
    "NotoSans.ttf":        "ofl/notosans/NotoSans%5Bwdth%2Cwght%5D.ttf",
    "PTSans-Regular.ttf":  "ofl/ptsans/PT_Sans-Web-Regular.ttf",
    "PTSans-Bold.ttf":     "ofl/ptsans/PT_Sans-Web-Bold.ttf",
    "PTSerif-Regular.ttf": "ofl/ptserif/PT_Serif-Web-Regular.ttf",
    # сжатые жирные  (стиль вывесок/шильдиков)
    "Oswald.ttf":          "ofl/oswald/Oswald%5Bwght%5D.ttf",
    "RobotoCondensed.ttf": "ofl/robotocondensed/RobotoCondensed%5Bwght%5D.ttf",
    "Caveat.ttf":          "ofl/caveat/Caveat%5Bwght%5D.ttf",
}
BASE = "https://raw.githubusercontent.com/google/fonts/main/"

for local_name, remote_path in FONT_URLS.items():
    out_path = CFG.FONTS_DIR / local_name
    if out_path.exists():
        continue  
    try:
        urllib.request.urlretrieve(BASE + remote_path, out_path)
        print(f"{local_name}")
    except Exception as e:
        # Скачивание шрифта не критично (сеть/зеркало могут быть недоступны, остальные шрифты всё равно скачаются)
        print(f"Не удалось скачать {local_name}: {e}")

# Шрифт, который гарантированно уже есть вместе с matplotlib и поддерживает кириллицу
import matplotlib
dejavu = Path(matplotlib.get_data_path()) / "fonts" / "ttf" / "DejaVuSans.ttf"
fallback_path = CFG.FONTS_DIR / "DejaVuSans.ttf"
if dejavu.exists() and not fallback_path.exists():
    import shutil
    shutil.copy(dejavu, fallback_path)
    print("DejaVuSans.ttf (fallback)")

print("Итого шрифтов:", len(list(CFG.FONTS_DIR.glob("*.ttf"))))

In [ ]:
# Открытый частотный словарь русского языка используется вместо жёстко заданного маленького списка слов, чтобы модель не привыкала к конкретным буквосочетаниям

url = "https://raw.githubusercontent.com/hermitdave/FrequencyWords/master/content/2018/ru/ru_50k.txt"
urllib.request.urlretrieve(url, "/tmp/ru_words.txt")

with open("/tmp/ru_words.txt", encoding="utf-8") as f:
    # Файл вида "слово частота" на строку — берём только само слово, топ-5000 по частоте
    RU_WORDS_LARGE = [line.split()[0] for line in f if line.strip()][:5000]

print(f"Загружено слов: {len(RU_WORDS_LARGE)}")

# Переопределение RU_WORDS большим списком, используется дальше в random_text()
RU_WORDS = RU_WORDS_LARGE

In [ ]:
import cv2

# Алфавиты и словари для генерации случайного текста
CYRILLIC_CHARS = "АБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯабвгдеёжзийклмнопрстуфхцчшщъыьэюя"
LATIN_CHARS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
DIGITS = "0123456789"
PUNCT = " .,-!?№/:%()"

# Запасной список русских слов (реально используется только если не выполнилась ячейка выше с частотным словарём RU_WORDS_LARGE, тогда RU_WORDS будет переопределён обратно на этот короткий список ниже
RU_WORDS = ["Продам", "Авито", "Куплю", "Цена", "Торг", "Доставка", "Новый", "Б/у",
            "Самовывоз", "Гарантия", "Скидка", "Объявление", "Состояние", "Звоните",
            "Отдам", "даром", "Размер", "Модель", "Телефон", "квартира", "срочно"]
EN_WORDS = ["SALE", "NEW", "PRICE", "DISCOUNT", "PREMIUM", "DELIVERY", "BONUS", "CALL"]


def random_text():
    """Генерирует случайную строку текста для рендеринга: смесь русских слов, английских слов, 
    числовых значений (цены/артикулы), случайных символьных последовательностей"""
    kind = random.random()
    if kind < 0.45:
        words = random.choices(RU_WORDS, k=random.randint(1, 3))
        text = " ".join(words)
    elif kind < 0.6:
        words = random.choices(EN_WORDS, k=random.randint(1, 2))
        text = " ".join(words)
    elif kind < 0.8:
        # числа (цены, артикулы) с возможной единицей измерения
        text = "".join(random.choices(DIGITS, k=random.randint(3, 10)))
        if random.random() < 0.5:
            text += random.choice([" руб.", " р.", "%", " шт."])
    else:
        # случайная символьная строка для устойчивости к "мусорному" OCR-выхлопу
        pool = CYRILLIC_CHARS + LATIN_CHARS + DIGITS + PUNCT
        text = "".join(random.choices(pool, k=random.randint(4, 14)))

    # Капс встречается очень часто на реальных вывесках, поэтому форсируем его в половине случаев
    if random.random() < 0.5:
        text = text.upper()

    return text


def load_fonts(fonts_dir, size_range=(18, 34)):
    """Собирает список путей ко всем .ttf/.otf файлам в папке шрифтов."""
    paths = list(Path(fonts_dir).glob("*.ttf")) + list(Path(fonts_dir).glob("*.otf"))
    return paths


FONT_PATHS = load_fonts(CFG.FONTS_DIR)
if len(FONT_PATHS) == 0:
    print("В папке fonts/ не найдено .ttf/.otf файлов, нужно добавить шрифты для синтетики.")


def get_random_font(size):
    """Возвращает случайный загруженный шрифт нужного размера; при ошибке загрузки происходит фоллбек на дефолтный шрифт PIL"""
    if FONT_PATHS:
        path = random.choice(FONT_PATHS)
        try:
            return ImageFont.truetype(str(path), size=size)
        except Exception:
            pass
    return ImageFont.load_default()


def random_bg(w, h):
    """Генерирует случайный фон для кропа (три равновероятных режима):
    1) Нейтральные объяления - почти однородный светлый фон с лёгким шумом;
    2) Баннеры/реклама - линейный градиент из двух случайных цветов;
    3) Вывески, домовые знаки - сплошной насыщенный цвет по фиксированной палитре тёмных оттенков, эта ветка добавлена по итогам визуального дебага self-consistency
       ошибок: без неё модель плохо работала на ярких/тёмных однотонных табличках."""
    mode = random.random()
    if mode < 0.35:
        base = np.random.randint(180, 256, size=(1, 1, 3), dtype=np.uint8)
        noise = np.random.randint(-15, 15, size=(h, w, 3))
        arr = np.clip(base.astype(int) + noise, 0, 255).astype(np.uint8)
    elif mode < 0.6:
        c1 = np.random.randint(120, 256, size=3)
        c2 = np.random.randint(120, 256, size=3)
        t = np.linspace(0, 1, w)[None, :, None]
        arr = (c1[None, None, :] * (1 - t) + c2[None, None, :] * t)
        arr = np.repeat(arr, h, axis=0).astype(np.uint8)
    else:
        # Сплошной насыщенный цвет
        palette = [
            (25, 90, 60),    # тёмно-зелёный 
            (139, 26, 26),   # тёмно-красный
            (60, 35, 25),    # тёмно-коричневый
            (20, 20, 20),    # почти чёрный
            (15, 40, 90),    # тёмно-синий
            (90, 15, 90),    # бордово-фиолетовый
        ]
        base_color = np.array(random.choice(palette))
        noise = np.random.randint(-10, 10, size=(h, w, 3))
        arr = np.clip(base_color[None, None, :] + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr, mode="RGB")


def add_texture_overlay(img):
    """Накладывает случайную текстуру (полосы/пятна/зерно) поверх изображения с малой непрозрачностью. Имитирует потёртые вывески, 
    ткань, металл, печать на упаковке"""
    w, h = img.size
    texture_type = random.choice(["noise_bands", "blobs", "grain"])

    overlay = Image.new("RGB", (w, h), (128, 128, 128))
    arr = np.array(overlay).astype(np.float32)
    if texture_type == "noise_bands":
        # горизонтальные полосы разной яркости
        bands = np.random.randint(-40, 40, size=(h, 1, 1))
        arr = arr + bands
    elif texture_type == "blobs":
        # случайные размытые пятна (потёртости/загрязнения)
        for _ in range(random.randint(3, 8)):
            cx, cy = random.randint(0, w), random.randint(0, h)
            r = random.randint(5, max(6, min(w, h) // 3))
            yy, xx = np.ogrid[:h, :w]
            mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2
            arr[mask] = arr[mask] + random.randint(-50, 50)
    else:  # grain - мелкозернистый шум по всей площади
        arr = arr + np.random.normal(0, random.uniform(15, 35), arr.shape)

    overlay = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    alpha = random.uniform(0.08, 0.25)  # лёгкое наложение - текст должен остаться читаемым
    return Image.blend(img, overlay, alpha)


def apply_emboss_effect(img):
    """Эффект тиснения/гравировки - похоже на металлические таблички"""
    emb = img.filter(ImageFilter.EMBOSS)
    alpha = random.uniform(0.3, 0.6)
    return Image.blend(img, emb, alpha)


def random_letter_spacing_stretch(img):
    """Случайное растяжение/сжатие по горизонтали имитирует condensed/expanded начертания шрифтов и геометрические искажения, которые вносит сам детектор текста
    при вырезании бокса."""
    w, h = img.size
    scale_x = random.uniform(0.7, 1.4)
    new_w = max(8, int(w * scale_x))
    img = img.resize((new_w, h), Image.BILINEAR)
    return img.resize((w, h), Image.BILINEAR)  


def draw_stylized_text(draw, position, text, font, base_color):
    """Рисует текст одним из трёх стилей: с обводкой (логотипы/вывески), с тенью, либо простым однотонным заполнением. Стиль выбирается случайно."""
    x, y = position
    style = random.random()

    if style < 0.25:
        stroke_color = tuple(np.random.randint(0, 255, size=3).tolist())
        stroke_width = random.randint(1, 3)
        draw.text((x, y), text, font=font, fill=base_color,
                   stroke_width=stroke_width, stroke_fill=stroke_color)
    elif style < 0.4:
        shadow_offset = random.randint(1, 3)
        shadow_color = tuple(max(0, c - 60) for c in base_color)
        draw.text((x + shadow_offset, y + shadow_offset), text, font=font, fill=shadow_color)
        draw.text((x, y), text, font=font, fill=base_color)
    else:
        draw.text((x, y), text, font=font, fill=base_color)


def contrast_text_color(bg_img):
    """Подбирает цвет текста, контрастный к средней яркости уже сгенерированного фона, без этого цвет текста выбирался бы
    независимо от фона и часто оказывался сливающимся"""
    arr = np.array(bg_img)
    mean_brightness = arr.mean()
    if mean_brightness < 128:
        return tuple(np.random.randint(200, 256, size=3).tolist())  # светлый текст на тёмном фоне
    else:
        return tuple(np.random.randint(0, 90, size=3).tolist())     # тёмный текст на светлом фоне


def random_perspective_warp(img, max_shift_frac=0.08):
    """Перспективная деформация имитирует небрежное фото под углом"""
    w, h = img.size
    arr = np.array(img)
    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    shift = max_shift_frac * min(w, h)
    dst = src + np.random.uniform(-shift, shift, src.shape).astype(np.float32)
    M = cv2.getPerspectiveTransform(src, dst)
    warped = cv2.warpPerspective(arr, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
    return Image.fromarray(warped)


def apply_motion_blur(img, max_kernel=7):
    """Смаз движения (горизонтальный или вертикальный) имитирует съёмку на телефон в движении."""
    arr = np.array(img)
    k = random.choice([3, 5, max_kernel])
    kernel = np.zeros((k, k))
    if random.random() < 0.5:
        kernel[k // 2, :] = 1.0  # горизонтальный смаз
    else:
        kernel[:, k // 2] = 1.0  # вертикальный смаз
    kernel /= k
    blurred = cv2.filter2D(arr, -1, kernel)
    return Image.fromarray(blurred)


def draw_watermark_text(img, text, font, alpha_range=(0.15, 0.45)):
    """Полупрозрачный текст поверх уже готового кропа (имитация водяного знака/наложенной надписи (частый паттерн на купюрах, скриншотах с наложенным текстом)"""
    overlay = Image.new("RGBA", img.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    color = tuple(np.random.randint(0, 255, size=3).tolist())
    alpha = int(255 * random.uniform(*alpha_range))
    bbox = draw.textbbox((0, 0), text, font=font)
    tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
    x = max(2, (img.width - tw) // 2)
    y = max(2, (img.height - th) // 2)
    draw.text((x, y), text, font=font, fill=color + (alpha,))
    return Image.alpha_composite(img.convert("RGBA"), overlay).convert("RGB")


def random_downsample_upsample(img, min_scale=0.25, max_scale=0.6):
    """Уменьшает изображение, затем возвращает к исходному размеру - имитирует ситуацию, когда реальный детектор текста находит очень маленький бокс, который потом апскейлится
    до входного разрешения модели (источник пикселизации на реальных данных)"""
    w, h = img.size
    scale = random.uniform(min_scale, max_scale)
    small = img.resize((max(4, int(w * scale)), max(4, int(h * scale))), Image.BILINEAR)
    return small.resize((w, h), random.choice([Image.NEAREST, Image.BILINEAR]))


def render_text_crop(img_h=CFG.IMG_H, img_w=CFG.IMG_W):
    """Главная функция генерации одного синтетического upright-кропа. Собирает вместе все функции выше: генерирует текст, потом фон, затем рисует стилизованный текст,
    после чего применяет цепочку случайных геометрических и фотографических искажений. Каждый эффект применяется с некоторой вероятностью"""
    text = random_text()
    # Размер шрифта - доля от высоты кропа, чтобы текст занимал бОльшую часть площади (на реальных вывесках текст обычно крупный, а не мелкий в углу)
    font_size = random.randint(int(img_h * 0.45), int(img_h * 0.9))
    font = get_random_font(font_size)

    img = random_bg(img_w, img_h)
    draw = ImageDraw.Draw(img)

    text_color = contrast_text_color(img)  # img уже содержит сгенерированный фон
    try:
        bbox = draw.textbbox((0, 0), text, font=font)
        tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
    except Exception:
        tw, th = font_size * len(text) // 2, font_size

    x = max(2, (img_w - tw) // 2 + random.randint(-5, 5))
    y = max(2, (img_h - th) // 2 + random.randint(-3, 3))
    draw_stylized_text(draw, (x, y), text, font, text_color)

    # геометрические искажения
    if random.random() < 0.3:
        img = random_letter_spacing_stretch(img)
    if random.random() < 0.3:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 1.2)))
    if random.random() < 0.35:
        angle = random.uniform(-6, 6)  # лёгкий наклон — НЕ 180°, ориентация остаётся "upright"
        img = img.rotate(angle, expand=False, fillcolor=(230, 230, 230))

    # текстурные эффекты 
    if random.random() < 0.15:
        img = apply_emboss_effect(img)
    if random.random() < 0.25:
        img = add_texture_overlay(img)

    # имитация условий съёмки
    if random.random() < 0.25:
        img = random_perspective_warp(img)
    if random.random() < 0.2:
        img = apply_motion_blur(img)

    # шум и сжатие
    if random.random() < 0.4:
        arr = np.array(img).astype(np.int16)
        noise = np.random.normal(0, random.uniform(2, 10), arr.shape)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        img = Image.fromarray(arr)
    if random.random() < 0.3:
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=random.randint(35, 80))
        buf.seek(0)
        img = Image.open(buf).convert("RGB")

    # дополнительные паттерны, добавленные по итогам визуального дебага 
    if random.random() < 0.15:
        img = draw_watermark_text(img, text, font)
    if random.random() < 0.25:
        img = random_downsample_upsample(img)

    return img


# визуальный тест одного сгенерированного кропа
sample_img = render_text_crop()
sample_img


Генерация и сохранение на диск N_SYNTH_SAMPLES upright-кропов (метку 0/180 будем применять уже в Dataset, чтобы не хранить на диске дублированные повёрнутые версии).

In [ ]:
def generate_synthetic_dataset(n_samples, out_dir):
    """Генерация и сохранение n_samples кропов в out_dir. Дозаписывает недостающие файлы, если часть уже сгенерирована (но если менялась логика render_text_crop, 
    старые файлы нужно удалить вручную перед перегенерацией)."""
    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True, parents=True)
    existing = len(list(out_dir.glob("*.jpg")))
    for i in tqdm(range(existing, n_samples), desc="Генерация синтетики"):
        img = render_text_crop()
        img.save(out_dir / f"synth_{i:06d}.jpg", quality=92)

generate_synthetic_dataset(CFG.N_SYNTH_SAMPLES, CFG.SYNTH_DIR)
print("Генерация завершилась. Файлов:", len(list(CFG.SYNTH_DIR.glob('*.jpg'))))

In [ ]:
# Визуальная проверка перед обучением: смотрим, что текст читаем, аугментации не превращают картинку в чистый шум.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(16, 4))
for ax in axes.flat:
    ax.imshow(render_text_crop())
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. `Dataset` с авто-разметкой ориентации и аугментациями

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def build_transforms(train=True):
    """Собирает пайплайн torchvision-трансформаций. Для train добавляются лёгкие фотометрические аугментации (ColorJitter, изредка блюр) и RandomErasing (заставляет
    модель опираться на всю картинку, а не на один локальный артефакт), для val эти аугментации не применяются, только ресайз и нормализация под ImageNet-статистику, 
    так как backbone предобучен на ImageNet)"""
    ops = [transforms.Resize((CFG.IMG_H, CFG.IMG_W))]
    if train:
        ops += [
            transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.03),
            transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.2),
        ]
    ops += [
        transforms.ToTensor(),
    ]
    if train:
        ops += [transforms.RandomErasing(p=0.2, scale=(0.02, 0.1))]
    ops += [transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)]
    return transforms.Compose(ops)


class OrientationDataset(Dataset):
    """Ключевой элемент авто-разметки: берёт upright-изображение и с веоятностью 0.5 переворачивает его на 180°, ставя соответствующую метку (1 - повёрнуто, 0 - нет). 
    Переворот происходит "на лету" в __getitem__, а не заранее, что экономит место на диске и гарантирует новую случайную выборку меток при каждой эпохе."""

    def __init__(self, image_paths, train=True):
        self.image_paths = image_paths
        self.train = train
        self.tf = build_transforms(train=train)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = Image.open(path).convert("RGB")

        label = 1 if random.random() < 0.5 else 0
        if label == 1:
            img = img.rotate(180)

        img = self.tf(img)
        return img, torch.tensor(label, dtype=torch.float32)

# Объединение синтетики в единый список путей
all_paths = list(CFG.SYNTH_DIR.glob("*.jpg"))
random.shuffle(all_paths)
print("Всего upright-изображений для обучения/валидации:", len(all_paths))

# Разбиение train/val по файлам (не по кропам с меткой, метка назначается динамически, поэтому утечки данных между train и val на уровне меток тут нет)
n_val = int(len(all_paths) * CFG.VAL_FRACTION)
val_paths = all_paths[:n_val]
train_paths = all_paths[n_val:]

train_ds = OrientationDataset(train_paths, train=True)
val_ds   = OrientationDataset(val_paths, train=False)

# Первичное создание loader-ов 
train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                           num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                           num_workers=CFG.NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")


## 5. Модель

**MobileNetV3-Small** (torchvision, ImageNet-претрейн) - компактная архитектура, голова заменена на один линейный выход (логит для sigmoid/BCEWithLogitsLoss)


In [ ]:

def build_model(backbone=CFG.BACKBONE, pretrained=True):
    """Создаёт модель backbone. Поддерживает mobilenet_v3_small (по умолчанию, основной вариант решения) и mobilenet_v3_large (запасной вариант большей ёмкости,
    на случай если small не хватит качества, результат использования описан в ячейке настроек)."""
    if backbone == "mobilenet_v3_small":
        weights = models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        net = models.mobilenet_v3_small(weights=weights)
        in_features = net.classifier[-1].in_features
        net.classifier[-1] = nn.Linear(in_features, 1)  #один логит вместо 1000 классов ImageNet
    elif backbone == "mobilenet_v3_large":
        weights = models.MobileNet_V3_Large_Weights.DEFAULT if pretrained else None
        net = models.mobilenet_v3_large(weights=weights)
        in_features = net.classifier[-1].in_features
        net.classifier[-1] = nn.Linear(in_features, 1)
    else:
        raise ValueError(f"Неизвестный backbone: {backbone}")
    return net

model = build_model().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Параметров в модели: {n_params:,}")


## 6. Обучение (AMP на CUDA)

In [ ]:
# Перегенерация синтетики "с нуля" - выполняется, если менялась логика render_text_crop. Старые файлы удаляются, чтобы в датасете не осталась смесь старой и новой
# версии синтетики. Если генератор не менялся с последнего запуска - эту ячейку можно пропустить
import shutil
shutil.rmtree(CFG.SYNTH_DIR, ignore_errors=True)
CFG.SYNTH_DIR.mkdir(exist_ok=True, parents=True)
generate_synthetic_dataset(CFG.N_SYNTH_SAMPLES, CFG.SYNTH_DIR)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS)
scaler = torch.amp.GradScaler(device=DEVICE.type, enabled=(DEVICE.type == "cuda"))


def smooth_labels(labels, smoothing=0.05):
    """Label smoothing: сдвигает жёсткие метки 0/1 к 0.025/0.975 вместо точных крайних значений. Не даёт модели становиться слишком самоуверенной на обучающих данных, это
    важно для домена, на котором модель не обучалась (реальные фото): без сглаживания модель на out-of-distribution картинках выдаёт уверенные, но случайные ответы 
    вместо честного "не знаю", что бьёт по Brier score. Применяется только к целям для loss на train - метрики (Brier/AUC) всегда считаются по настоящим 0/1 меткам"""
    return labels * (1 - smoothing) + 0.5 * smoothing


def run_epoch(loader, train=True):
    """Прогоняет одну эпоху обучения или валидации. Возвращает средний loss, Brier score, AUC, сырые логиты/метки (нужны для калибровки Platt scaling)."""
    model.train(train)
    total_loss, n = 0.0, 0
    all_logits, all_labels = [], []

    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train):
            with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                logits = model(imgs).squeeze(1)
                # На train применяем label smoothing к цели для loss; на val используем настоящие метки без сглаживания, val должен честно отражать реальное качество
                target = smooth_labels(labels) if train else labels
                loss = criterion(logits, target)

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * imgs.size(0)
        n += imgs.size(0)
        # В all_labels всегда пишем настоящие метки, метрики Brier/AUC должны считаться относительно истинных 0/1 значений
        all_logits.append(logits.detach().float().cpu())
        all_labels.append(labels.detach().float().cpu())

    all_logits = torch.cat(all_logits).numpy()
    all_labels = torch.cat(all_labels).numpy()
    probs = 1 / (1 + np.exp(-all_logits))  # sigmoid от логитов в вероятности
    brier = brier_score_loss(all_labels, probs)
    try:
        auc = roc_auc_score(all_labels, probs)
    except ValueError:
        auc = float("nan")  #может случиться, если в батче оказался только один класс
    return total_loss / n, brier, auc, all_logits, all_labels


best_val_brier = float("inf")
history = []

# DataLoader-ы создаются один раз до цикла обучения, чтобы persistent_workers=True давал эффект, воркеры переиспользуются между эпохами вместо пересоздания процессов заново каждый раз
train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                           num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True,
                           persistent_workers=(CFG.NUM_WORKERS > 0))
val_loader   = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                           num_workers=CFG.NUM_WORKERS, pin_memory=True,
                           persistent_workers=(CFG.NUM_WORKERS > 0))

# Early stopping: останавливаем обучение, если val Brier не улучшался patience эпох подряд 
patience = 4
epochs_no_improve = 0

for epoch in range(1, CFG.EPOCHS + 1):
    t0 = time.time()

    train_loss, train_brier, train_auc, _, _ = run_epoch(train_loader, train=True)
    val_loss, val_brier, val_auc, val_logits, val_labels = run_epoch(val_loader, train=False)
    scheduler.step()

    dt = time.time() - t0
    print(f"Epoch {epoch}/{CFG.EPOCHS} | "
          f"train_loss={train_loss:.4f} 1-Brier={1-train_brier:.4f} AUC={train_auc:.4f} | "
          f"val_loss={val_loss:.4f} 1-Brier={1-val_brier:.4f} AUC={val_auc:.4f} | {dt:.1f}s")

    history.append(dict(epoch=epoch, train_loss=train_loss, val_loss=val_loss,
                         val_brier=val_brier, val_auc=val_auc))

    if val_brier < best_val_brier:
        best_val_brier = val_brier
        epochs_no_improve = 0
        torch.save(model.state_dict(), CFG.PROJECT_DIR / "best_model.pt")
        # Сохраняем логиты/метки лучшей по val эпохи
        best_val_logits, best_val_labels = val_logits, val_labels
        print(f" Новый лучший val 1-Brier = {1-val_brier:.4f}, модель сохранена")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("Early stopping")
            break

print("Лучший val 1-Brier Score:", 1 - best_val_brier)

## 7. Калибровка вероятностей

Сырые sigmoid-вероятности нейросети обычно недокалиброваны. Поскольку метрика - Brier score, калибровка напрямую улучшает итоговый скор, особенно на "неоднозначных"
кропах (н-р, низкое качество), где модель должна давать значение ближе к 0.5, а не быть слишком уверенной.

Обучаем простую логистическую регрессию p_calibrated = sigmoid(a * logit + b) на части валидационных логитов, проверяем на другой части, оцениваем эффект
калибровки.

In [ ]:
# Загружаем веса лучшей по val эпохи 
model.load_state_dict(torch.load(CFG.PROJECT_DIR / "best_model.pt"))
model.eval()

# Делим val ещё раз пополам: на "calib-fit" и "calib-eval" (проверяем эффект калибровки на данных, которые калибровка не видела)
val_logits_arr = best_val_logits.reshape(-1, 1)
val_labels_arr = best_val_labels

n_calib = len(val_logits_arr) // 2
calib_fit_X, calib_eval_X = val_logits_arr[:n_calib], val_logits_arr[n_calib:]
calib_fit_y, calib_eval_y = val_labels_arr[:n_calib], val_labels_arr[n_calib:]

platt = LogisticRegression()
platt.fit(calib_fit_X, calib_fit_y)

raw_probs_eval = 1 / (1 + np.exp(-calib_eval_X.flatten()))
calibrated_probs_eval = platt.predict_proba(calib_eval_X)[:, 1]

brier_raw = brier_score_loss(calib_eval_y, raw_probs_eval)
brier_calibrated = brier_score_loss(calib_eval_y, calibrated_probs_eval)

print(f"1 - Brier (без калибровки):  {1 - brier_raw:.4f}")
print(f"1 - Brier (с калибровкой):   {1 - brier_calibrated:.4f}")

calib_params = {"a": float(platt.coef_[0][0]), "b": float(platt.intercept_[0])}
with open(CFG.PROJECT_DIR / "calibration.json", "w") as f:
    json.dump(calib_params, f)
print("Калибровка:", calib_params)


## 8. Инференс на тестовом датасете (20 000 боксов), получение submission.csv

In [ ]:
class TestDataset(Dataset):
    """Датасет для инференса на тестовых данных в отличие от OrientationDataset не переворачивает изображения и не генерирует метку (метка неизвестна и не должна
    быть известна по условию задания), возвращает картинку и её image_id."""
    def __init__(self, image_paths):
        self.image_paths = image_paths
        self.tf = build_transforms(train=False)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = Image.open(path).convert("RGB")
        img = self.tf(img)
        # .stem убирает расширение файла, т.к. в sample_submission.csv ("test_00000", а не "test_00000.png")
        return img, str(Path(path).stem)


# Собираем список файлов теста. rglob (рекурсивный поиск) вместо glob на случай, если архив распаковался с вложенной структурой папок (например test/images/*.png).
# Проверка обоих вариантов регистра расширения
test_paths = sorted(
    list(CFG.TEST_DIR.rglob("*.jpg")) +
    list(CFG.TEST_DIR.rglob("*.png")) +
    list(CFG.TEST_DIR.rglob("*.JPG")) +
    list(CFG.TEST_DIR.rglob("*.PNG"))
)
print("Тестовых кропов найдено:", len(test_paths))
if len(test_paths) > 0:
    print("Пример пути:", test_paths[0])
assert len(test_paths) > 0, "Тестовые боксы не найдены"

test_ds = TestDataset(test_paths)
test_loader = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                          num_workers=CFG.NUM_WORKERS, pin_memory=True)

model.eval()
image_ids, p180_list = [], []

with torch.no_grad():
    for imgs, names in tqdm(test_loader, desc="Инференс на тесте"):
        imgs = imgs.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = model(imgs).squeeze(1).float().cpu().numpy()

        # Применяем ту же калибровку, что откалибрована на валидации
        calibrated = platt.predict_proba(logits.reshape(-1, 1))[:, 1]

        image_ids.extend(names)
        p180_list.extend(calibrated.tolist())

submission = pd.DataFrame({"image_id": image_ids, "p_180": p180_list})
submission["p_180"] = submission["p_180"].clip(0.0, 1.0)  # на всякий случай гарантируем диапазон [0, 1]
submission.to_csv(CFG.OUT_SUBMISSION, index=False)
print("Сохранено:", CFG.OUT_SUBMISSION, "| строк:", len(submission))
submission.head()

In [ ]:
# Сверка итогового submission с sample_submission.csv перед сдачей: проверка, что все image_id из эталона присутствуют и нет лишних или пропущенных id.
sample = pd.read_csv(CFG.TEST_CSV)
sub_check = pd.read_csv(CFG.OUT_SUBMISSION)

print("Строк в sample:", len(sample), "| строк в submission:", len(sub_check))

missing = set(sample["image_id"]) - set(sub_check["image_id"])
extra   = set(sub_check["image_id"]) - set(sample["image_id"])
print("Не хватает id:", len(missing))
print("Лишних id:", len(extra))

if len(missing) == 0 and len(extra) == 0:
    print("Полное совпадение по image_id — формат корректен")
else:
    print("Примеры несовпадений:", list(missing)[:5], list(extra)[:5])

## Валидация на реальных данных без разметки: self-consistency проверка

Стандартная валидация выше считается на синтетике, она быстро выходит на неоправданно высокие показатели (AUC примерно 0.999), не отражающие качество на реальных фото с их
доменным сдвигом (другие шрифты, освещение, текстуры, сжатие).

По условию задания ручная разметка тестовых данных запрещена, поэтому для адекватной оценки качества на реальном тесте без единой размеченной метки используется self-consistency проверка: для случайной подвыборки реальных тестовых кропов считается вероятность p1 для оригинала и p2 для его же версии, повёрнутой на 180°. У корректно работающей модели должно выполняться p1 + p2 = 1 (примерно 1, конечно).

Эта метрика (а не val на синтетике) использовалась как главный ориентир при итеративном улучшении генератора синтетики в разделе: собирался батч кропов с наибольшей ошибкой
согласованности, изучался визуально, усиливалась синтетика под замеченный паттерн, модель переобучалась → метрика перепроверялась.

In [ ]:

import matplotlib.pyplot as plt

model.eval()
# Фиксированный размер выборки (6000) даёт стандартную ошибку оценки доли 0.6%, этого хватает, чтобы уверенно отличать реальный прогресс между итерациями от шума измерения
sample_test_paths = random.sample(test_paths, min(6000, len(test_paths)))
diffs = []
probs_orig, probs_flipped = [], []

with torch.no_grad():
    for path in tqdm(sample_test_paths, desc="Проверка согласованности"):
        img = Image.open(path).convert("RGB")
        img_flipped = img.rotate(180)

        x1 = build_transforms(train=False)(img).unsqueeze(0).to(DEVICE)
        x2 = build_transforms(train=False)(img_flipped).unsqueeze(0).to(DEVICE)

        logit1 = model(x1).squeeze().item()
        logit2 = model(x2).squeeze().item()

        p1 = platt.predict_proba([[logit1]])[0, 1]
        p2 = platt.predict_proba([[logit2]])[0, 1]

        probs_orig.append(p1)
        probs_flipped.append(p2)
        diffs.append(abs((p1 + p2) - 1.0))  # хотелось бы p1 + p2 == 1

print(f"Средняя ошибка согласованности |p1+p2-1|: {np.mean(diffs):.4f}")
print(f"Доля кропов с ошибкой > 0.2: {np.mean(np.array(diffs) > 0.2):.2%}")

plt.hist(diffs, bins=50)
plt.title("Распределение ошибки согласованности p(orig) + p(180°) - 1")
plt.xlabel("|p1 + p2 - 1|")
plt.show()

### Гистограмма распределения предсказаний по всему тесту

Дополнительный sanity-check: смотрим, что распределение p_180 на всех 20 000 боксах бимодальное (пики у 0 и 1, модель в основном уверена), но с разумной долей значений около 0.5 (неоднозначные кропы всё-таки есть), а не сплошной провал в середину, не крайности без сомнений 

In [ ]:
plt.hist(submission["p_180"], bins=50)
plt.title("Распределение p_180 на test.zip")
plt.xlabel("p_180")
plt.show()

print("Доля предсказаний в [0.4, 0.6] (неуверенные):",
      ((submission["p_180"] > 0.4) & (submission["p_180"] < 0.6)).mean())

In [ ]:

# Визуальный дебаг: отбираем кропы с наибольшей self-consistency ошибкой и смотрим на них, чтобы понять, какой визуальный паттерн модель систематически не улавливает
# и усилить его представленность в синтетике 
results = []
with torch.no_grad():
    for path in tqdm(sample_test_paths):
        img = Image.open(path).convert("RGB")
        img_flipped = img.rotate(180)
        x1 = build_transforms(train=False)(img).unsqueeze(0).to(DEVICE)
        x2 = build_transforms(train=False)(img_flipped).unsqueeze(0).to(DEVICE)
        logit1 = model(x1).squeeze().item()
        logit2 = model(x2).squeeze().item()
        p1 = platt.predict_proba([[logit1]])[0, 1]
        p2 = platt.predict_proba([[logit2]])[0, 1]
        results.append((path, p1, p2, abs((p1 + p2) - 1.0)))

results.sort(key=lambda r: -r[3])  # худшие (по ошибке согласованности) первые

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, (path, p1, p2, err) in zip(axes.flat, results[:12]):
    ax.imshow(Image.open(path))
    ax.set_title(f"p={p1:.2f} err={err:.2f}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9. Калибровка уверенности (shrink к 0.5)


После первой отправки submission.csv был получен результат 1 - Brier = 0.70082 (Brier = 0.29918). Распределение предсказаний p_180 на тесте оказалось почти бимодальным
(большинство значений близко к 0 или 1) - для таких "почти бинарных" предсказаний Brier score приближённо равен доле неправильных ответов, то есть реальная точность модели на тесте - около 70%.

Для предсказаний, близких к крайним значениям (0 или 1), при фактической точности q, ожидаемый Brier ≈ 1 - q. Но оптимальное значение предсказания при точности q - это
p* = 1 - q, а не строго 0/1. При q ≈ 0.7 оптимальный Brier составляет q * (1-q) = 0.21, что лучше, чем 1-q = 0.3, получаемый при излишне уверенных предсказаниях.

Вывод: модель слишком уверена относительно своей фактической точности на реальном тестовом домене (это ожидаемо, калибровка Platt scaling обучалась на синтетическом val, где модель почти всегда права).

### Метод

Рассматривается семейство линейных сжатий предсказаний к центру:
p'(s) = 0.5 + (p - 0.5) · s, где s ∈ [0, 1]

s = 1 - без изменений (исходные предсказания), при s стремящкмся к 0 все предсказания стремятся к 0.5.

Раскладывая Brier(s) = mean((p'(s) - y)²) через d = p - 0.5 и учитывая, что для бинарных меток y из {0, 1} величина (y - 0.5)² ≡ 0.25 всегда, получаем квадратичную по s функцию:

Brier(s) = A·s² - 2B·s + C, где:
A = mean((p - 0.5)²) считается напрямую из submission, без меток
C = 0.25 - константа для любых бинарных меток
B = (A + C - Brier(s=1)) / 2 - выражается через уже известный отправленный результат


Минимум параболы:

s* = B / A


Этот расчёт не использует метки отдельных примеров, только один агрегированный числовой результат уже отправленного решения.

### Ещё одна проверка

Результат сверен независимым способом: на синтетическом валидационном наборе (где настоящие метки известны) искусственно занижена точность до уровня, аналогичного тестовому
(~70%, случайной порчей части меток), и на этих "испорченных" данных численно подобран оптимальный shrink_factor полным перебором. Оба метода дали близкие значения:

| Метод | s* |
|---|---|
| Точная формула (по реальному Brier=0.29918) | 0.383 |
| Эмуляция пониженной точности на val | 0.40 |

Близкое совпадение независимых методов подтверждает, что оценка не случайна.

### Результат

После применения s* = 0.383 к отправленным предсказаниям: 1 - Brier = 0.77290 (Brier = 0.22710) против исходных 0.70082 без калибровки. Прогнозируемое по формуле
значение (Brier ≈ 0.211) оказалось чуть оптимистичнее фактического, расхождение говорит о том, что самоуверенность модели, судя по всему, распределена неравномерно (часть предсказаний действительно надёжна, часть почти случайна), что линейное сжатие одним общим коэффициентом может скорректировать лишь частично.

In [ ]:
import numpy as np
import pandas as pd

# Загружаем уже отправленный submission.csv (Brier=0.29917657)
submission = pd.read_csv(CFG.OUT_SUBMISSION)
p = submission["p_180"].values

# Расчёт оптимального shrink_factor по формуле
# A и C вычисляются без меток; brier_1 - фактический результат уже отправленного решения
A = np.mean((p - 0.5) ** 2)
C = 0.25
brier_1 = 0.29917657  # официальный результат первой отправки

B = (A + C - brier_1) / 2
s_star_formula = B / A
predicted_min_brier = C - B**2 / A

print(f"A (разброс предсказаний от 0.5): {A:.4f}")
print(f"Оптимальный shrink_factor (формула): s* = {s_star_formula:.4f}")
print(f"Прогнозируемый минимальный Brier при этом s*: {predicted_min_brier:.4f}")
print(f"Прогнозируемый 1-Brier: {1 - predicted_min_brier:.4f}")


# Проверка: эмуляция пониженной точности на синтетическом val 
def shrink_probabilities(p, shrink_factor):
    """Сжимает вероятности к 0.5, то есть снижает избыточную уверенность модели.
    shrink_factor=1.0 - без изменений. shrink_factor=0.0 - все предсказания превращаются в 0.5."""
    return 0.5 + (p - 0.5) * shrink_factor

val_probs_calibrated = platt.predict_proba(best_val_logits.reshape(-1, 1))[:, 1]

# Портим часть меток val, чтобы эмулировать точность, близкую к реальной тестовой (~70%)
np.random.seed(123)
corrupt_frac = 0.30
corrupted_labels = best_val_labels.copy()
flip_idx = np.random.choice(len(corrupted_labels), size=int(len(corrupted_labels) * corrupt_frac), replace=False)
corrupted_labels[flip_idx] = 1 - corrupted_labels[flip_idx]

best_shrink, best_brier_emul = None, float("inf")
for shrink in np.arange(0.1, 1.05, 0.05):
    shrunk = shrink_probabilities(val_probs_calibrated, shrink)
    b = brier_score_loss(corrupted_labels, shrunk)
    if b < best_brier_emul:
        best_brier_emul = b
        best_shrink = shrink

print(f"\nНезависимая проверка (эмуляция ~70% точности на val):")
print(f"shrink_factor = {best_shrink:.2f}, Brier на эмуляции = {best_brier_emul:.4f}")

# Берём точную формулу как основную 
FINAL_SHRINK_FACTOR = s_star_formula  # 0.383

print(f"\nИтоговый FINAL_SHRINK_FACTOR = {FINAL_SHRINK_FACTOR:.3f}")

# Применяем калибровку и формируем финальный submission.csv
submission_final = submission.copy()
submission_final["p_180"] = shrink_probabilities(submission["p_180"].values, FINAL_SHRINK_FACTOR)
submission_final["p_180"] = submission_final["p_180"].clip(0.0, 1.0)
submission_final.to_csv(CFG.OUT_SUBMISSION, index=False)

print("\nРаспределение предсказаний после калибровки:")
print(submission_final["p_180"].describe())


## 9. Итоговое описание решения

### Задача и метрика
Классификация ориентации текстового кропа (0°/180°) с выдачей вероятности p_180. Метрика - 1 - Brier Score, что требует не только правильного класса, но и калиброванной уверенности в предсказании.

### Подход
Данные для обучения не предоставлялись, поэтому решение построено на синтетически сгенерированных данных: рендерим текст в правильной (upright) ориентации, а метку 0/180
создаём автоматически случайным переворотом на этапе загрузки батча (50/50 баланс классов).

### 1. Генерация синтетических данных
Кастомный генератор на PIL/OpenCV, рендерящий текстовые кропы с широким разнообразием визуальных условий, приближенных к реальным фото вывесок:
- **Текст**: частотные русские слова (топ-5000 из открытого словаря [hermitdave/FrequencyWords](https://github.com/hermitdave/FrequencyWords), MIT), английские
  слова/фразы, числовые строки, случайные символьные последовательности, часто капс.
- **Шрифты**: около 20 открытых шрифтов с Google Fonts (SIL Open Font License): обычные, жирные, сжатые, рукописные, декоративные.
- **Фон**: однородный светлый, градиент, сплошной насыщенный цвет (стиль вывесок).
- **Стилизация текста**: обводка, тень, водяной знак; цвет подбирается контрастно к фону.
- **Геометрические искажения**: наклон, перспектива, растяжение или сжатие.
- **Фотографические искажения**: блюр, motion blur, JPEG-сжатие, шум, тиснение, текстурные наложения и тд

### 2. Модель
MobileNetV3-Small (torchvision, ImageNet-претрейн), один линейный выход (1,518,881 параметров) 

### 3. Обучение
Вход 64×256. BCEWithLogitsLoss + label smoothing (0.05). AdamW, cosine LR schedule, AMP на CUDA, early stopping. Аугментации на train: ColorJitter, GaussianBlur, RandomErasing.
Random seed зафиксирован (SEED=42) для random, numpy, torch.

### 4. Калибровка
Platt scaling поверх логитов, обученная на отложенной части валидации улучшает Brier score, особенно на неоднозначных кропах.

### 5. Валидация 
Стандартная hold-out валидация на синтетике не репрезентативна для реального теста (быстро выходит на AUC примерно 0.999, что не отражает доменный сдвиг). Используется self-consistency проверка: для реальных тестовых кропов без меток считается p(orig) и p(rotate180), у корректной модели p1 + p2 ≈ 1. Итеративный процесс:
худшие по этой метрике кропы, потом визуальный анализ паттерна, усиление синтетики, переобучение, перепроверка. За несколько итераций средняя ошибка согласованности снижена с ~0.49 до ~0.22-0.25, доля кропов с ошибкой >0.2 — с ~63% до ~33-38%.

### 6. Что было в попытках, но не вошло в финальное решение
- Mixup между случайными парами кропов в батче - по self-consistency проверке не дал улучшения (видимо, смешивание двух семантически несвязанных кропов текста не создаёт
  осмысленной "промежуточной" метки ориентации). Убран из финального пайплайна.
- Реальные датасеты (ICDAR2019-MLT, MIDV-500) рассматривались как источник доменного разнообразия, но ICDAR-MLT не содержит кириллицы (Arabic/Bangla/Chinese/Devanagari/
  English/French/German/Italian/Japanese/Korean), MIDV-500 оказался недоступен из-за сетевых ограничений окружения (FTP-порт заблокирован файрволом). Решение полностью основано
  на синтетике с усиленными аугментациями.

### 7. Использованные open-source компоненты
- torchvision — веса MobileNetV3-Small (ImageNet-претрейн)
- Шрифты Google Fonts (SIL Open Font License), DejaVu Sans 
- Частотный словарь русского языка: [hermitdave/FrequencyWords](https://github.com/hermitdave/FrequencyWords) (MIT)
- scikit-learn для калибровки

Внешние API не использовались. LLM/VLM в решении не использовались.